In [1]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.1 MB/s eta 0:00:00


In [6]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ==========================================
# 1. Preparando a Base de Conhecimento
# ==========================================
documentos = [
    "O Google Colab é um ambiente de notebooks Jupyter gratuito que roda na nuvem e oferece acesso a GPUs.",
    "RAG significa Retrieval-Augmented Generation. É uma técnica que melhora as respostas de modelos de IA fornecendo fatos externos.",
    "A biblioteca Transformers da Hugging Face fornece milhares de modelos pré-treinados para processamento de linguagem natural.",
    "A linguagem de programação Python foi criada por Guido van Rossum e lançada pela primeira vez em 1991.",
    "O pão de queijo é uma receita típica do estado de Minas Gerais, no Brasil, feito com polvilho e queijo."
]

# ==========================================
# 2. Carregando os Modelos da Hugging Face
# ==========================================
print("Carregando o modelo de Embeddings (busca)...")
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print("Carregando o LLM (geração)...")
gerador = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    max_new_tokens=100,
    max_length=None,  # <-- CORREÇÃO 2: Remove o aviso chato da tela
    temperature=0.1,
    do_sample=True
)

# ==========================================
# 3. Criando o Banco de Dados Vetorial (FAISS)
# ==========================================
embeddings_docs = embedder.encode(documentos)
faiss.normalize_L2(embeddings_docs)
dimensao = embeddings_docs.shape[1]
index = faiss.IndexFlatIP(dimensao)
index.add(embeddings_docs)

# ==========================================
# 4. Função Principal do RAG
# ==========================================
# <-- CORREÇÃO 1: top_k=1 força a trazer APENAS a frase exata
def fazer_pergunta_rag(pergunta, top_k=1):

    vetor_pergunta = embedder.encode([pergunta])
    faiss.normalize_L2(vetor_pergunta)
    distancias, indices = index.search(vetor_pergunta, top_k)

    contexto_recuperado = "\n".join([documentos[i] for i in indices[0]])

    # <-- CORREÇÃO 3: Prompt muito mais rigoroso ("anti-alucinação")
    prompt = (
        f"Você é um assistente estritamente factual. Responda à pergunta usando EXCLUSIVAMENTE o contexto abaixo.\n"
        f"Se a informação não estiver no contexto, não invente dados.\n\n"
        f"Contexto:\n{contexto_recuperado}\n\n"
        f"Pergunta: {pergunta}\n\n"
        f"Resposta:"
    )

    mensagens = [{"role": "user", "content": prompt}]

    resultado = gerador(mensagens, return_full_text=False)
    resposta_final = resultado[0]['generated_text'].strip()

    print("-" * 60)
    print(f"**Pergunta:** {pergunta}")
    print(f"**Contexto Recuperado:**\n{contexto_recuperado}")
    print(f"\n**Resposta da IA:**\n{resposta_final}")
    print("-" * 60)

# ==========================================
# 5. Testando a Aplicação
# ==========================================
fazer_pergunta_rag("O que significa a sigla RAG?")
fazer_pergunta_rag("De onde é o pão de queijo?")
fazer_pergunta_rag("Quem inventou o Python?")

Carregando o modelo de Embeddings (busca)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Carregando o LLM (geração)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'max_length', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


------------------------------------------------------------
**Pergunta:** O que significa a sigla RAG?
**Contexto Recuperado:**
RAG significa Retrieval-Augmented Generation. É uma técnica que melhora as respostas de modelos de IA fornecendo fatos externos.

**Resposta da IA:**
A sigla RAG (Retrieval-Augmented Generation) significa "Retrieval-Augmented Generation". Isso refere-se ao uso de informações adicionais para melhorar a qualidade das respostas produzidas por modelos de inteligência artificial (IA). Em outras palavras, RAG é uma técnica que ajuda os sistemas de aprendizado de máquina a extraer e incorporar mais informações em suas respostas, permitindo que essas respostas sejam mais relevant
------------------------------------------------------------
------------------------------------------------------------
**Pergunta:** De onde é o pão de queijo?
**Contexto Recuperado:**
O pão de queijo é uma receita típica do estado de Minas Gerais, no Brasil, feito com polvilho e queijo.
